# Unitree G1 Colab GPU IK Benchmark

This notebook validates a hardware-free Unitree G1 arm IK workflow on a Colab GPU. It downloads the G1 URDF from `unitreerobotics/xr_teleoperate`, extracts the torso-to-palm chains, and solves batched wrist targets under the actual joint limits.

In [ ]:
REPO_URL = "https://github.com/zack-dev-cm/unitree-g1-colab-ik.git"
!rm -rf /content/unitree-g1-colab-ik
!git clone --depth 1 {REPO_URL} /content/unitree-g1-colab-ik
%pip -q install -e /content/unitree-g1-colab-ik

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable Runtime > Change runtime type > GPU before running this notebook."
print("torch", torch.__version__)
print("cuda", torch.version.cuda)
print("gpu", torch.cuda.get_device_name(0))

In [ ]:
from unitree_colab_ik import run_benchmark

results = run_benchmark(batch_size=384, steps=240, device="cuda")
for result in results:
    print(
        f"{result.side:5s} mean={result.mean_error_m:.5f}m "
        f"p95={result.p95_error_m:.5f}m max={result.max_error_m:.5f}m "
        f"success={result.success_rate:.3f} "
        f"limit_violation={result.limit_violation_rad:.2e} "
        f"throughput={result.targets_per_s:.1f} targets/s"
    )

In [ ]:
import matplotlib.pyplot as plt

labels = [result.side for result in results]
mean_cm = [result.mean_error_m * 100 for result in results]
p95_cm = [result.p95_error_m * 100 for result in results]
max_cm = [result.max_error_m * 100 for result in results]

x = range(len(labels))
width = 0.24
plt.figure(figsize=(7, 4))
plt.bar([i - width for i in x], mean_cm, width=width, label="mean")
plt.bar(list(x), p95_cm, width=width, label="p95")
plt.bar([i + width for i in x], max_cm, width=width, label="max")
plt.axhline(2.0, color="tab:red", linestyle="--", linewidth=1, label="2 cm target")
plt.xticks(list(x), labels)
plt.ylabel("position error (cm)")
plt.title("Unitree G1 batched wrist IK error")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.show()

In [ ]:
for result in results:
    assert result.success_rate >= 0.98, result
    assert result.mean_error_m <= 0.01, result
    assert result.limit_violation_rad <= 1e-6, result

print("PASS: Unitree G1 Colab GPU IK benchmark passed for both arms.")